# WBC 2026 Spray Charts with baseball-field-viz

This notebook visualizes WBC 2026 batted ball data using **[baseball-field-viz](https://github.com/yasumorishima/baseball-field-viz)** — a Python library for drawing baseball fields and spray charts from Statcast data.

### What baseball-field-viz enables
- `draw_field(ax)` — draws a baseball field on any matplotlib Axes
- `transform_coords(df)` — converts Statcast `hc_x`/`hc_y` to feet (home plate at origin)
- `spraychart(ax, df, color_by='events')` — one-liner spray chart

Unlike pybaseball's built-in `spraychart()`, this library gives direct access to the matplotlib `Axes`, so you can overlay **heatmaps** (seaborn `kdeplot`, `histplot`) on top.

### Dataset
- **WBC 2026 Scouting** — Statcast data for all 18 WBC 2026 countries (2024–2025 MLB regular season)
- Dataset: [yasunorim/wbc-2026-scouting](https://www.kaggle.com/datasets/yasunorim/wbc-2026-scouting)

### WBC 2026 Scouting Dashboard
An interactive scouting dashboard built from this dataset is available at:  
**https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/**

---

In [ ]:
!pip install baseball-field-viz -q

import os
import numpy as np
import pandas as pd
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from baseball_field_viz import transform_coords, draw_field, spraychart

print("baseball-field-viz ready")

In [ ]:
DATA_DIR = "/kaggle/input/wbc-2026-scouting"

df = pd.read_csv(f"{DATA_DIR}/statcast_batters.csv")
rosters = pd.read_csv(f"{DATA_DIR}/rosters.csv")

print(f"Statcast records : {len(df):,}")
print(f"Countries        : {df['country_name'].nunique()}")
print(f"Players          : {df['player_name'].nunique()}")
print(f"Batted balls     : {df['hc_x'].notna().sum():,}")
print()

hit_events = ['home_run', 'double', 'triple', 'single']
hits_by_country = (
    df[df['events'].isin(hit_events)]
    .groupby('country_name')
    .size()
    .sort_values(ascending=False)
)
print("Hits by country:")
print(hits_by_country.to_string())

## 1. All WBC Countries — Spray Chart Overview

Hits (single / double / triple / home run) from all 18 WBC 2026 countries, colored by country.

`draw_field(ax)` draws the field, then we manually scatter each country with its own color.

In [ ]:
hits = df[
    df['hc_x'].notna() & df['events'].isin(hit_events)
].copy()
hits = transform_coords(hits)

country_list = sorted(hits['country_name'].unique())
colors = cm.tab20(np.linspace(0, 1, len(country_list)))
color_map = dict(zip(country_list, colors))

fig, ax = plt.subplots(figsize=(12, 12))
draw_field(ax)

for country in country_list:
    subset = hits[hits['country_name'] == country]
    ax.scatter(
        subset['x'], subset['y'],
        c=[color_map[country]], alpha=0.35, s=12,
        label=f"{country} ({len(subset)})"
    )

ax.set_xlim(-350, 350)
ax.set_ylim(-50, 420)
ax.set_xlabel("X (feet)")
ax.set_ylabel("Y (feet)")
ax.set_title("WBC 2026 — All Countries Batted Balls (Hits Only)", fontsize=14)
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

USA, Dominican Republic, and Venezuela account for the largest share of hits — reflecting the high number of MLB players on those rosters. Despite differing sample sizes, batted ball distributions tend to follow similar patterns across countries, concentrating in center and right-center field.

## 2. Top 4 Countries — Spray Chart Comparison

Using `spraychart(ax, df, color_by='events')` for each country.  
Points are colored: **home run = red**, **triple = orange**, **double = blue**, **single = green**.

In [ ]:
top_countries = ['USA', 'Dominican Republic', 'Venezuela', 'Japan']

fig, axs = plt.subplots(2, 2, figsize=(16, 14))

for ax, country in zip(axs.flat, top_countries):
    df_c = df[df['country_name'] == country]
    n_hits = len(
        df_c[df_c['events'].isin(hit_events) & df_c['hc_x'].notna()]
    )
    spraychart(ax, df_c, color_by='events', title=f"{country}  ({n_hits} hits)")

plt.suptitle("WBC 2026 — Spray Charts by Country", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

Home runs (red) spread to all fields across every country. Japan's chart tends to show more contact to right-center, which may reflect tendencies to go the other way against MLB pitching — though the smaller sample size warrants caution in interpretation.

## 3. Japan — Hits vs Outs Heatmap

This is where `baseball-field-viz` shines over pybaseball's built-in `spraychart()`:  
since we have direct access to the `Axes` object, we can overlay seaborn's `kdeplot` to show **density distributions**.

- **Left**: Hits heatmap (single / double / triple / home run)
- **Right**: Outs heatmap (all other batted balls)

In [ ]:
df_jpn = df[df['country_name'] == 'Japan']
df_jpn_t = transform_coords(df_jpn[df_jpn['hc_x'].notna()])

hits_jpn = df_jpn_t[df_jpn_t['events'].isin(hit_events)]
outs_jpn = df_jpn_t[~df_jpn_t['events'].isin(hit_events)]

fig, axs = plt.subplots(1, 2, figsize=(16, 8))

draw_field(axs[0])
sns.kdeplot(
    data=hits_jpn, x='x', y='y', ax=axs[0],
    cmap='Reds', fill=True, alpha=0.6, levels=10
)
axs[0].set_xlim(-350, 350)
axs[0].set_ylim(-50, 400)
axs[0].set_title(f"Japan — Hits Heatmap  (n={len(hits_jpn)})", fontsize=13)

draw_field(axs[1])
sns.kdeplot(
    data=outs_jpn, x='x', y='y', ax=axs[1],
    cmap='Blues', fill=True, alpha=0.6, levels=10
)
axs[1].set_xlim(-350, 350)
axs[1].set_ylim(-50, 400)
axs[1].set_title(f"Japan — Outs Heatmap  (n={len(outs_jpn)})", fontsize=13)

plt.suptitle("Japan — Batted Ball Distribution", fontsize=15)
plt.tight_layout()
plt.show()

The heatmap reveals where Japan's batters tend to make contact. Outs are concentrated more in the infield and shallow outfield, while hits spread further. Comparing the two panels helps identify zones where Japan's batters are productive — the kind of insight that's difficult to see from a plain scatter plot.

## 4. Home Run Distribution — Top 5 Countries

Home runs only, for the five countries with the most HRs in the dataset.

In [ ]:
hr_counts = (
    df[df['events'] == 'home_run']
    .groupby('country_name')
    .size()
    .sort_values(ascending=False)
    .head(5)
)
top5_hr_countries = hr_counts.index.tolist()

fig, axs = plt.subplots(1, 5, figsize=(28, 7))

for ax, country in zip(axs, top5_hr_countries):
    df_c = df[(df['country_name'] == country) & (df['events'] == 'home_run')]
    n = len(df_c[df_c['hc_x'].notna()])
    spraychart(ax, df_c, color_by='events', title=f"{country}\n({n} HR)")
    # remove legend for compact display
    ax.get_legend().remove()

plt.suptitle("WBC 2026 — Home Run Spray Charts (Top 5 Countries)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Home run patterns are broadly similar across countries — most land in center to left-center field — reflecting the pull tendencies common among power hitters in MLB. Individual country tendencies are worth exploring, but sample size differences make direct comparisons tentative.

## Explore More: WBC 2026 Scouting Dashboard

For an interactive view of each player's stats by country, check out the **WBC 2026 Scouting Dashboard**:

**https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/**

The dashboard covers all 18 WBC 2026 countries with per-player batting and pitching metrics based on 2024–2025 MLB Statcast data.

---

## Resources

| | Link |
|---|---|
| baseball-field-viz (PyPI) | https://pypi.org/project/baseball-field-viz/ |
| baseball-field-viz (GitHub) | https://github.com/yasumorishima/baseball-field-viz |
| WBC 2026 Scouting Dataset | https://www.kaggle.com/datasets/yasunorim/wbc-2026-scouting |
| WBC 2026 Scouting Dashboard | https://wbc-2026-scouting-dashboard-zvg.caffeine.xyz/ |

```python
pip install baseball-field-viz
```

```python
from baseball_field_viz import draw_field, spraychart, transform_coords

fig, ax = plt.subplots(figsize=(10, 10))
spraychart(ax, df, color_by='events')
plt.show()
```